# Step 2: Universe node (halo positions)

**Predictions written BEFORE running on real data:**
1. Over ~1.5-11 Mpc/h, halo clustering should give correlation dimension around 2 (+/- 0.3, our calibration error), not 3.
2. The local slope should rise toward 3 as scale increases (approach to homogeneity). If it stays flat near 2 across all scales, that would be surprising.
3. The Poisson control in the same box must come back at 3.0. If not, the pipeline is broken.

Improvement over Step 1: TNG boxes are periodic, so we use periodic pair counting (no edge bias).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

rng = np.random.default_rng(42)

# ---- PLUG IN YOUR DATA HERE ---------------------------------------------
# positions: (N, 3) array of halo/subhalo centers, in the same units as BOX
# BOX: side length of the periodic simulation box (TNG100-1 is 75 Mpc/h = 75000 kpc/h)
# Use your existing TNG loader from the C9 repo, or load a saved array:
#   positions = np.load("tng100_halo_positions.npy"); BOX = 75000.0
positions = None
BOX = 75000.0
N_MAX = 8000   # keep the N the same as the calibration; take the most massive N_MAX halos if you have masses
# --------------------------------------------------------------------------

## Periodic correlation dimension with scale-dependent slope

In [ ]:
def corr_integral_periodic(pts, box, radii):
    pts = np.mod(pts, box)
    tree = cKDTree(pts, boxsize=box)
    n = len(pts)
    c = tree.count_neighbors(tree, radii)
    return (c - n) / (n * (n - 1))

def local_slopes(radii, c, width=4):
    lr, lc = np.log(radii), np.log(c)
    mids, sl = [], []
    for i in range(len(lr) - width + 1):
        mids.append(np.exp(lr[i:i + width].mean()))
        sl.append(np.polyfit(lr[i:i + width], lc[i:i + width], 1)[0])
    return np.array(mids), np.array(sl)

def analyse(pts, box, r_lo=0.02, r_hi=0.15, n_r=16):
    radii = np.logspace(np.log10(r_lo * box), np.log10(r_hi * box), n_r)
    c = corr_integral_periodic(pts, box, radii)
    ok = c > 0
    radii, c = radii[ok], c[ok]
    d_global = np.polyfit(np.log(radii), np.log(c), 1)[0]
    mids, sl = local_slopes(radii, c)
    return d_global, radii, c, mids, sl

def subsample_std(pts, box, frac=0.8, reps=15):
    out = []
    for _ in range(reps):
        sub = pts[rng.choice(len(pts), int(frac * len(pts)), replace=False)]
        out.append(analyse(sub, box)[0])
    return float(np.std(out))

## Controls (run these first; they need no real data)

In [ ]:
def neyman_scott(n, box, n_parents=40, sigma_frac=0.02):
    parents = rng.random((n_parents, 3)) * box
    idx = rng.integers(0, n_parents, n)
    return parents[idx] + rng.normal(scale=sigma_frac * box, size=(n, 3))

controls = {
    "Poisson (must be ~3.0)": rng.random((N_MAX, 3)) * BOX,
    "Clustered, not fractal": neyman_scott(N_MAX, BOX),
}
results = {}
for name, pts in controls.items():
    d, radii, c, mids, sl = analyse(pts, BOX)
    results[name] = (mids, sl)
    print(f"{name:28s} D = {d:.3f}  (subsample std {subsample_std(pts, BOX):.3f})  local slopes {sl.min():.2f} to {sl.max():.2f}")

## Real data

In [ ]:
if positions is None:
    print("No data loaded yet. Set `positions` above, then re-run this cell.")
else:
    pos = np.asarray(positions)[:N_MAX]
    d, radii, c, mids, sl = analyse(pos, BOX)
    results["REAL halos"] = (mids, sl)
    print(f"REAL halos: N={len(pos)}  D = {d:.3f}  (subsample std {subsample_std(pos, BOX):.3f})")
    print(f"Local slope runs {sl[0]:.2f} (small r) -> {sl[-1]:.2f} (large r)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, (mids, sl) in results.items():
    ax.semilogx(mids / BOX, sl, marker="o", label=name)
ax.axhline(3, ls=":", c="grey"); ax.axhline(2, ls=":", c="grey")
ax.set_xlabel("r / box size"); ax.set_ylabel("local correlation dimension")
ax.legend(); ax.set_title("Dimension vs scale (universe node)")
plt.tight_layout(); plt.savefig("universe_node.png", dpi=150); plt.show()

## Reading the result
- Compare REAL against both controls. Poisson is the "no structure" floor (3.0); clustered is the "structure but not fractal" check (its slope drifts hard with scale).
- A fractal claim needs a flat local slope across at least a decade of scale. Otherwise report it as scale-dependent clustering, which is what the cosmic web is expected to be.
- Write down the result against the three predictions above, whatever it is.